# SNC — Selective Sound Removal Web App

Launches a Gradio web app for the conditioned separator: upload an
audio or video file, detect the sounds in it, choose which to remove,
and compare before/after.

Needs a trained `separator_unet_film_multi_v2.1.h5` on Drive (produced by
`colab_train_conditioned_separator.ipynb`). A GPU runtime is faster
but not required.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
DRIVE_ROOT = '/content/drive/MyDrive/snc'
GITHUB_USER = 'keremtutumlu'
GITHUB_REPO_NAME = 'selective-noise-cancelling'
BRANCH = 'feature/separator-quality-overhaul'

# Private repo? Add a Colab Secret named GITHUB_TOKEN (key icon, scope: repo).
import os, subprocess, shutil
from pathlib import Path

token = None
try:
    from google.colab import userdata
    token = userdata.get('GITHUB_TOKEN')
except Exception:
    pass

clone_url = (f'https://{token}@github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git'
             if token else
             f'https://github.com/{GITHUB_USER}/{GITHUB_REPO_NAME}.git')

REPO_DIR = Path(f'/content/{GITHUB_REPO_NAME}')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
result = subprocess.run(['git', 'clone', '-b', BRANCH, clone_url, str(REPO_DIR)],
                        capture_output=True, text=True)
if result.returncode != 0:
    print('STDERR:', result.stderr)
    raise RuntimeError('git clone failed — if the repo is private add a '
                       'GITHUB_TOKEN secret in the left sidebar.')
os.chdir(REPO_DIR)
print('Working dir:', os.getcwd())

In [ ]:
# Symlink saved_models to Drive so the trained model is available.
drive_models = Path(DRIVE_ROOT) / 'saved_models'
local_models = REPO_DIR / 'saved_models'
if local_models.is_symlink() or local_models.exists():
    local_models.unlink() if local_models.is_symlink() else shutil.rmtree(local_models)
local_models.symlink_to(drive_models)

ckpt = drive_models / 'separation_models' / 'separator_unet_film_multi_v2.1.h5'
assert ckpt.exists(), f'Trained model not found at {ckpt} — run training first.'
print('Model found:', ckpt)

In [ ]:
!pip install -q gradio librosa==0.11.0 soundfile

## Launch the app

This cell stays running. Open the **public URL** it prints (ends in
`.gradio.live`) to use the app.

In [ ]:
!python src/application/webapp.py